# Revised JGR Fig.3/4/9/10 from saved MLR comparison tables
CSV-only renderer. This notebook never refits OLS or importance diagnostics.
Restart Kernel → Run All independently after the computation notebook has saved passing QC.
Builds four residualized-only figures; exports only revised Fig.4/10 v5 PNG/PDF, leaving Fig.3/9 v1 untouched, only when genuine inputs have been computed.
Inspect real figures after export for text/data occlusion; automatic extent checks are not a scientific visual review.

## 1. Paths and publication style

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
# Open this notebook with the repository as the Jupyter working directory,
# or set MLR_CODE_ROOT to the cloned repository path before starting Jupyter.
CODE_ROOT = Path(os.environ.get("MLR_CODE_ROOT", str(Path.cwd()))).expanduser().resolve()
if not (CODE_ROOT / "mlr_comparison_core.py").is_file():
    raise FileNotFoundError(f"Set CODE_ROOT to the repository containing the helper modules: {CODE_ROOT}")
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
import mlr_comparison_core as core
INPUT_ROOT = Path(os.environ.get("MLR_INPUT_ROOT", "/data1/huangy/fig6/NC/v12/four_init_full_pipeline_NCVIconcurrent"))
OUTPUT_ROOT = Path(os.environ.get("MLR_OUTPUT_ROOT", str(INPUT_ROOT / "mlr_raw_residual_comparison")))
print("INPUT_ROOT:", INPUT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Initializations:", core.INITS, "Inclusive BJT windows:", core.WINDOWS)
core.output_guard(INPUT_ROOT, OUTPUT_ROOT)
import matplotlib.pyplot as plt
import mlr_comparison_plotting as plotting
# Fail before plotting when a previous Jupyter kernel cached an older
# module, or when CODE_ROOT silently points to an older repository clone.
expected_core=(CODE_ROOT / "mlr_comparison_core.py").resolve()
expected_plotting=(CODE_ROOT / "mlr_comparison_plotting.py").resolve()
actual_core=Path(core.__file__).resolve()
actual_plotting=Path(plotting.__file__).resolve()
print("CODE_ROOT:", CODE_ROOT)
print("Loaded MLR core from:", actual_core)
print("Loaded MLR plotting from:", actual_plotting)
if actual_core != expected_core or actual_plotting != expected_plotting:
    raise RuntimeError(
        "MLR helper module path mismatch; an old copy is being imported. "
        "Use the PR #15 feature-branch checkout and Restart Kernel → Run All. "
        f"Expected core={expected_core}, loaded={actual_core}; "
        f"expected plotting={expected_plotting}, loaded={actual_plotting}"
    )
if (getattr(plotting, "FIGURE_VERSION", None) != "residualized" or
        getattr(plotting, "FIGURE_REVISION", None) != "lofo_deltaR2_v1" or
        getattr(plotting, "IMPORTANCE_REVISION", None) != "lofo_deltaR2_v5"):
    raise RuntimeError(
        "Stale MLR plotting module detected: PR #15 requires "
        "FIGURE_VERSION=residualized, FIGURE_REVISION=lofo_deltaR2_v1 and IMPORTANCE_REVISION=lofo_deltaR2_v5. "
        f"Loaded version={getattr(plotting, 'FIGURE_VERSION', None)!r}, "
        f"revision={getattr(plotting, 'FIGURE_REVISION', None)!r}. "
        "Update the PR #15 checkout, then Restart Kernel → Run All. "
        "Do not proceed with the old normalized-LOFO plotting code."
    )
print("MLR figure code revision verified:", plotting.FIGURE_REVISION)
print("Fixed width:", plotting.WIDTH, "in; PNG dpi:", plotting.DPI)

## 2. Read QC-approved numerical tables and create shared axes/order

In [ ]:
data, computation_manifest = plotting.load_results(OUTPUT_ROOT)
plan = plotting.figure_plan(data)
print("Computation QC:", computation_manifest["computational_status"])
print("Historical reproduction:", computation_manifest["history_status"])
print("Shared method limits:", plan["bars"])
print("Residualized-only paper figures; raw parameters are kept for model-history audits.")

## 3. Fig.3 Total and Fig.9 14–17 June: four initializations

In [ ]:
scatter_figures = plotting.make_figures(data, plan, "scatter")
for item in scatter_figures:
    print(item["qc"]["basename"], item["qc"]["status"])

## 4. Fig.4 Total and Fig.10 14–17 June: LMG/Johnson/LOFO

In [ ]:
importance_figures = plotting.make_figures(data, plan, "importance")
for item in importance_figures:
    print(item["qc"]["basename"], item["qc"]["status"])

## 5. Export ONLY revised Fig.4 and Fig.10 PNG/PDF pairs after rendered layout checks

In [ ]:
built = importance_figures  # Style-only v5: Fig3/Fig9 remain available as previously exported v1
figure_manifest = plotting.save_figures(built, INPUT_ROOT, OUTPUT_ROOT)
print(figure_manifest.to_string(index=False))

## 6. File, canvas, axes and text QC; manual review

In [ ]:
assert len(figure_manifest) == 2
assert (figure_manifest.png_width_px == 3900).all()
assert (figure_manifest.status == "PASS").all()
print("Two revised importance PNG and PDF pairs:", OUTPUT_ROOT / "figures")
print("Axes/text QC:", OUTPUT_ROOT / "qc/figure_lofo_deltaR2_v5_qc.json")
print("Review genuine figures for annotation/data overlap, readability at 6.5 inch, and captions in mlr_comparison_captions.md.")
print("No 1:1 lines, significance annotations, causal interpretation, or difference heatmaps were added.")